# Tutorial: Visualizing FalkorDB Graphs with PyGraphistry

This tutorial demonstrates how to connect to FalkorDB, query graph data using Cypher, and visualize the results using PyGraphistry.

FalkorDB is a high-performance graph database built on Redis, providing blazing-fast query execution for graph workloads.

## Setup

### Install Dependencies

In [ ]:
# Uncomment to install dependencies
# !pip install --user pandas
# !pip install --user graphistry[falkordb]
# Or install FalkorDB separately:
# !pip install --user graphistry falkordb

### Import Libraries

In [ ]:
import pandas as pd
import graphistry
from falkordb import FalkorDB

# Test imports
print(f"Graphistry version: {graphistry.__version__}")
print(f"Pandas version: {pd.__version__}")

### Configure Graphistry

Register with your Graphistry server credentials:

In [ ]:
# For Graphistry Hub (cloud)
graphistry.register(
    api=3,
    username='YOUR_USERNAME',
    password='YOUR_PASSWORD'
)

# For self-hosted Graphistry
# graphistry.register(
#     api=3,
#     protocol='https',
#     server='your.graphistry.server',
#     username='YOUR_USERNAME',
#     password='YOUR_PASSWORD'
# )

## Connect to FalkorDB

Configure your FalkorDB connection:

In [ ]:
# Connect to FalkorDB
FALKORDB_CONFIG = {
    'host': 'localhost',
    'port': 6379,
    'password': 'your_password'  # Optional
}

db = FalkorDB(**FALKORDB_CONFIG)

# Register FalkorDB with Graphistry
graphistry.register(api=3, username='YOUR_USERNAME', password='YOUR_PASSWORD', falkordb=db)

print("Connected to FalkorDB!")
print(f"Available graphs: {db.list_graphs()}")

## Method 1: Using Global Registration

In [ ]:
# Query a graph using the globally registered connection
g = graphistry.falkordb_cypher(
    'social',  # Graph name
    """
    MATCH (person:Person)-[knows:KNOWS]->(friend:Person)
    RETURN person, knows, friend
    LIMIT 100
    """
)

# Plot the results
g.plot()

## Method 2: Using Method Chaining

In [ ]:
# Create a new FalkorDB connection and query in one go
db2 = FalkorDB(host='localhost', port=6379, password='your_password')

g = (graphistry
     .falkordb(db2)
     .falkordb_cypher('social', 'MATCH (n:Person) RETURN n LIMIT 50')
     .bind(point_color='type')
     .plot())

## Parameterized Queries

Use parameterized queries for safer, more flexible queries:

In [ ]:
query = """
MATCH (person:Person)-[knows:KNOWS]-(friend:Person)
WHERE person.name = $name
RETURN person, knows, friend
"""

params = {"name": "Alice"}

g = graphistry.falkordb_cypher('social', query, params)

# Inspect the results
print("Nodes:")
print(g._nodes.head())
print("\nEdges:")
print(g._edges.head())

# Visualize
g.plot()

## Advanced Visualization

Customize your visualization with PyGraphistry's rich encoding options:

In [ ]:
g = graphistry.falkordb_cypher(
    'social',
    """
    MATCH (person:Person)-[r:KNOWS|FOLLOWS]->(other:Person)
    RETURN person, r, other
    LIMIT 200
    """
)

# Customize visualization
g2 = (g
      .bind(point_title='name', point_color='type')
      .bind(edge_color='type')
      .encode_point_size('degree', categorical_mapping={'low': 50, 'medium': 100, 'high': 200})
      .settings(url_params={'play': 1000}))

g2.plot()

## Working with Multiple Graphs

FalkorDB supports multiple named graphs in a single database:

In [ ]:
# Query from different graphs
social_graph = graphistry.falkordb_cypher('social', 'MATCH (n) RETURN n LIMIT 50')
product_graph = graphistry.falkordb_cypher('products', 'MATCH (n) RETURN n LIMIT 50')

print(f"Social graph has {len(social_graph._nodes)} nodes")
print(f"Product graph has {len(product_graph._nodes)} nodes")

## Creating Test Data

If you need to create test data in FalkorDB:

In [ ]:
# Create a test graph
graph = db.select_graph('test_social')

# Create nodes and relationships
result = graph.query("""
    CREATE (alice:Person {name: 'Alice', age: 30, city: 'New York'})
    CREATE (bob:Person {name: 'Bob', age: 25, city: 'San Francisco'})
    CREATE (charlie:Person {name: 'Charlie', age: 35, city: 'Seattle'})
    CREATE (alice)-[:KNOWS {since: 2015}]->(bob)
    CREATE (bob)-[:KNOWS {since: 2018}]->(charlie)
    CREATE (alice)-[:FOLLOWS]->(charlie)
    RETURN alice, bob, charlie
""")

print(f"Created {result.nodes_created} nodes and {result.relationships_created} relationships")

# Now visualize it
g = graphistry.falkordb_cypher('test_social', 'MATCH (n)-[r]->(m) RETURN n, r, m')
g.plot()

## Summary

This tutorial covered:

1. Connecting to FalkorDB from PyGraphistry
2. Running Cypher queries against FalkorDB graphs
3. Visualizing query results with PyGraphistry
4. Using parameterized queries
5. Customizing visualizations
6. Working with multiple graphs

For more information:
- [FalkorDB Documentation](https://docs.falkordb.com)
- [PyGraphistry Documentation](https://pygraphistry.readthedocs.io)
- [PyGraphistry GitHub](https://github.com/graphistry/pygraphistry)